# Amber Molecular Dynamics Workflow Guide

**Version:** 1.0.0  
**Created:** 2026-04-06  
**Last Updated:** 2026-04-07  
**Tested on:** 2026-04-07

**Audience:** Computational chemists and biophysicists running protein simulations, binding free energy calculations, and biomolecular MD workflows on HPC systems

**System:** HPC cluster with CPU/GPU nodes, Amber license required, MPI support via environment modules  
**Modules required:** `amber` (pmemd.cuda, pmemd.MPI.cuda) + `ambertools` (tleap, sander, sander.MPI, cpptraj, antechamber)


## Executive Summary

This guide explains **how to run Amber molecular dynamics simulations on this HPC cluster** using a staged workflow that takes you from raw structure to analyzed trajectory.

It covers:
- Quick 5-minute setup and first simulation
- CPU vs GPU decision-making
- Complete `sander.MPI` and `pmemd.cuda` parallelization guidance
- Real workflow examples (energy minimization, NVT heating, NPT production MD)
- Step-by-step troubleshooting

The goal is to **get from zero to a running simulation in 15 minutes**, then optimize for production runs.

**Key workflow:** `tleap` → `sander` (minimize) → `sander.MPI` / `pmemd.cuda` (heat + produce) → `cpptraj` (analyze)


## Table of Contents

- [Executive Summary](#executive-summary)
- [PART I — Getting Started (Read Once)](#part-i--getting-started-read-once)
  - [TL;DR: 5-Minute Quick Start](#tldr-5-minute-quick-start)
  - [Step 1: Verify Your Environment](#step-1-verify-your-environment)
  - [Step 2: Prepare Your Input Files](#step-2-prepare-your-input-files)
  - [Step 3: Submit Your First Job](#step-3-submit-your-first-job)
  - [Step 4: Monitor Progress](#step-4-monitor-progress)
- [PART II — Understanding Amber (Read Carefully)](#part-ii--understanding-amber-read-carefully)
  - [What Amber Does](#what-amber-does)
  - [CPU vs GPU Decision Tree](#cpu-vs-gpu-decision-tree)
  - [Parallelization: sander.MPI, pmemd.cuda, and OpenMP](#parallelization-sandermpi-pmemdcuda-and-openmp)
  - [Workflow Pipeline Stages](#workflow-pipeline-stages)
- [PART III — Real Workflows (Use As Needed)](#part-iii--real-workflows-use-as-needed)
  - [Configuration Variables](#configuration-variables)
  - [Directory and Input File Setup](#directory-and-input-file-setup)
  - [Workflow 1: Single-Node CPU Simulation (Build Partition)](#workflow-1-single-node-cpu-simulation-build-partition)
  - [Workflow 2: GPU-Accelerated Simulation](#workflow-2-gpu-accelerated-simulation)
  - [Workflow 3: Multi-Node Production Run](#workflow-3-multi-node-production-run)
- [PART IV — Troubleshooting (Skim When Broken)](#part-iv--troubleshooting-skim-when-broken)
  - [Monitoring Your Job](#monitoring-your-job)
  - [Common Errors and Fixes](#common-errors-and-fixes)
  - [Understanding Output Files](#understanding-output-files)
  - [FAQ](#faq)

_💡 Tip: Enable nbextensions like "Collapsible Headings" and "Table of Contents (2)" for better navigation._


---

## PART I — Getting Started (Read Once)

**Purpose:** "I just want this to work."

This part gets you from zero to a running Amber simulation in minimal time. No deep understanding needed yet—just follow the steps.


### TL;DR: 5-Minute Quick Start

```bash
# 1. Prepare directories and write tleap script
mkdir -p input mdin_files
cp your_protein.pdb input/
# (write leaprc.input — see Step 2)

# 2. Generate and submit the job script (CPU, build partition)
cat > amber.sbatch << 'SBATCH'
#!/bin/bash
#SBATCH --job-name=amber-cpu
#SBATCH --partition=build
#SBATCH --nodes=1
#SBATCH --ntasks=16
#SBATCH --mem=32G
#SBATCH --time=08:00:00
#SBATCH --output=amber_%j.out
module load amber ambertools
tleap -f leaprc.input
sander -O -i mdin_files/em.mdin -o em.mdout -p system.prmtop -c system.inpcrd -r em.rst7
mpirun -np $SLURM_NTASKS sander.MPI -O -i mdin_files/heat.mdin -o heat.mdout -p system.prmtop -c em.rst7 -r heat.rst7 -x heat.nc
mpirun -np $SLURM_NTASKS sander.MPI -O -i mdin_files/prod.mdin -o prod.mdout -p system.prmtop -c heat.rst7 -r prod.rst7 -x prod.nc
SBATCH
sbatch amber.sbatch

# 3. Check status
squeue -u $USER
```

**Expected result:** Job accepts; output appears in the working directory with `.mdout`, `.rst7`, and `.nc` files.


### Step 1: Verify Your Environment

Confirm that both modules are loaded and all key tools are in your PATH before doing anything else:


In [ ]:
%%bash
# Load Amber modules and verify all key tools are available
# amber     → pmemd.cuda, pmemd.MPI.cuda (licensed GPU engines)
# ambertools → tleap, sander, sander.MPI, antechamber, cpptraj (free tools)
module load amber ambertools 2>/dev/null
amber.python --version 2>/dev/null | head -3

echo ""
echo "=== Core tools ==="
for tool in tleap sander sander.MPI antechamber cpptraj pmemd.cuda; do
    command -v $tool &>/dev/null && echo "  OK  $tool" || echo "  --  $tool (not found)"
done


**Expected output:**
```
Python 3.11.x :: Amber distribution

=== Core tools ===
  OK  tleap
  OK  sander
  OK  sander.MPI
  OK  antechamber
  OK  cpptraj
  OK  pmemd.cuda
```

If `tleap` or `sander` show `not found`, run `module load ambertools` before retrying.  
If `pmemd.cuda` shows `not found`, run `module load amber` on a GPU node.


### Step 2: Prepare Your Input Files

Create a minimal alanine residue PDB file — the canonical Amber test case — for pipeline validation:


In [ ]:
%%bash
# Create a minimal single-residue alanine PDB (Amber standard test structure)
mkdir -p input

cat > input/ala.pdb << 'EOF'
ATOM      1  N   ALA A   1       1.201   0.847   0.000  1.00  0.00           N
ATOM      2  CA  ALA A   1       0.000   0.000   0.000  1.00  0.00           C
ATOM      3  C   ALA A   1      -1.250   0.880   0.000  1.00  0.00           C
ATOM      4  O   ALA A   1      -1.170   2.090  -0.003  1.00  0.00           O
ATOM      5  CB  ALA A   1       0.073  -0.774   1.303  1.00  0.00           C
ATOM      6  H   ALA A   1       1.200   1.431   0.818  1.00  0.00           H
ATOM      7  HA  ALA A   1       0.052  -0.602  -0.905  1.00  0.00           H
ATOM      8  HB1 ALA A   1       1.091  -1.167   1.401  1.00  0.00           H
ATOM      9  HB2 ALA A   1      -0.596  -1.611   1.302  1.00  0.00           H
ATOM     10  HB3 ALA A   1      -0.215  -0.125   2.152  1.00  0.00           H
TER
END
EOF

echo "Created test PDB file (alanine):"
ls -lah input/ala.pdb


**Expected output:**
```
Created test PDB file (alanine):
-rw-r--r-- 1 user group 782 Apr  6 10:00 input/ala.pdb
```

Continue to Step 3 below to submit this test structure. To run a full workflow with your own protein, see Workflow 1 in Part III.


### Step 3: Submit Your First Job

Generate the required input files inline and submit a minimal energy-minimization job for the alanine test structure created in Step 2:


In [ ]:
%%bash
# STEP 3: Generate input files inline and submit a test minimization job
# Runs: tleap (system prep) → sander (energy minimization) on the alanine test structure
# Full 3-stage workflows (minimize → heat → produce) are in Part III.

module load amber ambertools 2>/dev/null

# Create mdin files directory and a minimal energy minimization control file
mkdir -p mdin_files

cat > mdin_files/em_quick.mdin << 'EOF'
Amber quick energy minimization (Part I test)
 &cntrl
  imin   = 1,       ! Minimize (not MD)
  maxcyc = 5000,    ! 5,000 steps
  ncyc   = 2500,    ! Steepest descent for first half, CG for second
  ntb    = 1,       ! Periodic boundary, constant volume
  ntp    = 0,
  cut    = 8.0,
  ntpr   = 100,
  ntwr   = 5000,
 /
EOF

# Create tleap input using the alanine PDB created in Step 2
# Force field: ff99SBildn  Water: SPC/E
cat > leaprc_quick.input << 'EOF'
source leaprc.protein.ff99SBildn
source leaprc.water.spce
mol = loadpdb input/ala.pdb
check mol
solvateBox mol SPCBOX 10.0
addions mol Na+ 0
saveamberparm mol system.prmtop system.inpcrd
savepdb mol system.pdb
quit
EOF

# Generate and submit the SLURM job script
cat > amber_test.sbatch << 'ENDJOB'
#!/bin/bash
#SBATCH --job-name=amber-test-em
#SBATCH --partition=build
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --mem=8G
#SBATCH --time=00:30:00
#SBATCH --output=amber_test_%j.out
#SBATCH --error=amber_test_%j.err
module load amber ambertools
tleap -f leaprc_quick.input
sander -O \
  -i mdin_files/em_quick.mdin -o em_test.mdout \
  -p system.prmtop -c system.inpcrd -r em_test.rst7
echo "✅ Test minimization complete."
ENDJOB

echo "=== Submitting test minimization job ==="
sbatch amber_test.sbatch


**Expected output:**
```
=== Submitting test minimization job ===
Submitted batch job 12345
```

The job writes `amber_test_12345.out` (standard output) and `em_test.mdout` (Amber energy log) when complete. Continue to Step 4 to check its status.


### Step 4: Monitor Progress

Check your job status:


In [ ]:
%%bash
echo "=== Your Amber Jobs ==="
squeue -u $USER --format="%.7i %.9P %.20j %.8u %.2t %.10M %.6D %R"

**Expected output:**
```
=== Your Amber Jobs ===
  JOBID PARTITION                 NAME     USER  ST       TIME  NODES
  12345     build          amber_cpu_run   user   R       5:12      1
```

**Looking for:**
- `ST=R` (Running) — simulation is executing
- `ST=PD` (Pending) — waiting for resources
- Job not listed — already completed (check `sacct -u $USER`)

✅ **Success!** Your simulation is running. Continue to Part II to understand optimization, or jump to Part III for advanced workflows.

**Next Steps:**
- **Want to understand sander vs pmemd and MPI scaling?** Continue to [Part II](#part-ii--understanding-amber-read-carefully) below.
- **Ready for production workflows?** Jump to [Part III](#part-iii--real-workflows-use-as-needed).
- **Something failed?** Go to [Part IV troubleshooting](#part-iv--troubleshooting-skim-when-broken).


---

## PART II — Understanding Amber (Read Carefully)

**Purpose:** "Help me not screw this up."

This part teaches you about Amber's architecture, when to use CPU vs GPU, and how parallelization actually works.


### What Amber Does

Amber (**Assisted Model Building with Energy Refinement**) is a molecular dynamics suite specialized for biomolecular simulations of proteins, nucleic acids, and small molecules.

**Key capabilities:**

| Task | Tool | Purpose |
|------|------|---------|
| Structure preparation | `tleap` | Convert PDB → topology + coordinates; assign force field; solvate |
| Small molecule parameters | `antechamber` | Generate GAFF force fields for non-standard residues or ligands |
| Energy minimization | `sander` | Remove steric clashes before dynamics |
| CPU molecular dynamics | `sander.MPI` | Parallel MD on CPU nodes via MPI |
| GPU molecular dynamics | `pmemd.cuda` | GPU-accelerated MD (~50x faster than CPU) |
| Trajectory analysis | `cpptraj` | RMSD, clustering, hydrogen bonds, RMSF, solvent analysis |
| Binding free energy | `MMPBSA.py` | MM-PBSA and MM-GBSA free energy estimates |

The standard pipeline runs these stages sequentially, each using the previous stage's output.


### CPU vs GPU Decision Tree

```
How many atoms in your system?
│
├─ < 5,000 atoms
│   └─ Use CPU only (GPU setup overhead dominates runtime)
│       └─ sander or sander.MPI on build partition
│
├─ 5,000 – 100,000 atoms
│   └─ GPU strongly recommended (50x speedup typical)
│       ├─ GPU available? → pmemd.cuda (–-partition=gpu)
│       └─ CPU only? → sander.MPI (build partition, 16–32 cores)
│
└─ > 100,000 atoms
    └─ GPU required or multi-node CPU
        ├─ GPU available? → pmemd.cuda (single card) or pmemd.MPI.cuda
        └─ No GPU? → Multi-node sander.MPI (4+ nodes, 32+ cores)
```


### Parallelization: sander.MPI, pmemd.cuda, and OpenMP

The workflow uses different executables depending on resources—here's what's happening:

| Component | SLURM Flag | What It Does | Example |
|-----------|-----------|-------------|---------|
| **sander** | (none) | Single-core CPU; minimization and testing | `sander -O -i em.mdin ...` |
| **sander.MPI** | `--ntasks=N` | Multi-core CPU via MPI | `mpirun -np 16 sander.MPI -O ...` |
| **pmemd.cuda** | `--gres=gpu:1` | Single GPU; fastest path for production | `pmemd.cuda -O -i prod.mdin ...` |
| **pmemd.MPI.cuda** | `--ntasks=N --gres=gpu:N` | Multi-GPU via MPI (when available) | `mpirun -np 2 pmemd.MPI.cuda ...` |

**Recommended rule:** Use `sander` for minimization (robust, always safe), then switch to `pmemd.cuda` for heating and production when GPU is available.

**Practical limits:**
- `sander.MPI` scales well to ~32 ranks; diminishing returns beyond that
- `pmemd.cuda` uses 1 GPU by default; multi-GPU needs `pmemd.MPI.cuda`
- OpenMP threading is not the primary parallelization strategy in Amber


### Host Selection Guide (Authoritative Reference)

Use this table to match your simulation to the right partition and resource configuration:

| Partition | System Size | Simulation Length | CPU Cores | GPU | Typical Request | Performance (ns/day) |
|-----------|---|---|---|---|---|---|
| **build** | < 10K atoms | ≤ 100 ns | 4–16 | ❌ None | `--ntasks=8 --cpus-per-task=1` | ~5 ns/day |
| **build** | 10–50K atoms | ≤ 50 ns | 16–32 | ❌ None | `--ntasks=16 --cpus-per-task=1` | ~2 ns/day |
| **gpu** | 5–50K atoms | ≤ 1 μs | 1–4 | ✅ 1× | `--ntasks=1 --cpus-per-task=4 --gres=gpu:1` | ~200 ns/day |
| **gpu** | 50–100K atoms | ≤ 500 ns | 1–4 | ✅ 1× | `--ntasks=1 --cpus-per-task=4 --gres=gpu:1` | ~50 ns/day |
| **gpu** | > 100K atoms | ≤ 100 ns | 4–8 | ✅ 2× | `--ntasks=2 --cpus-per-task=4 --gres=gpu:2` | ~20 ns/day |
| **build** | any | **analysis** | 4–16 | ❌ None | `--ntasks=8 --cpus-per-task=1` | (post-processing) |

**Decision Tree:**
```
Do I have > 100,000 atoms?
  YES  → GPU required or multi-node (Workflow 3)
  NO   → Is system 5–50K atoms?
           YES  → GPU strongly preferred (Workflow 2, ~50–200× faster than CPU)
           NO   → System < 5K atoms; CPU is fine (Workflow 1)
```

**Key Rules:**
- Always run energy minimization with `sander` (serial, most robust)
- Switch to `pmemd.cuda` for heating and production when GPU is available
- For CPU-only runs: 16 ranks is the practical ceiling for most systems


### Workflow Pipeline Stages

Here's what happens when you submit an Amber job:

```
[1] SUBMIT               sbatch → job script receives SLURM environment
    ├─ Load modules (amber)
    ├─ Set environment (CUDA_VISIBLE_DEVICES, MPI options)
    └─ Detect input files

[2] SYSTEM PREPARATION
    ├─ tleap → system.prmtop (topology) + system.inpcrd (coordinates)
    └─ (optional) antechamber → GAFF parameters for non-standard residues

[3] EXECUTE STAGES SEQUENTIALLY
    ├─ Stage 1: sander      (minimization: em.mdin → em.rst7)
    ├─ Stage 2: sander.MPI / pmemd.cuda  (heating NVT: heat.mdin → heat.rst7)
    └─ Stage 3: sander.MPI / pmemd.cuda  (production NPT: prod.mdin → prod.nc)

[4] OUTPUT
    ├─ Save *.mdout (energy, temperature, pressure log)
    ├─ Save *.nc (NetCDF trajectory — main simulation output)
    ├─ Save *.rst7 (restart file for job continuation)
    └─ Report timing + completion status
```


---

## PART III — Real Workflows (Use As Needed)

**Purpose:** "Show me how to actually use this."

Copy these workflows and adapt them to your system.


### Configuration Variables

Before running production workflows, set these variables **once** for all subsequent templates:


In [ ]:
%%bash
# Set environment variables for all Amber workflows
# Customize these for your project

export PROJECT_NAME="my_amber_project"
export INPUT_PDB="ala.pdb"
export WORK_DIR="${PWD}/amber_work"
export MPI_RANKS=16
export FORCE_FIELD="ff99SBildn"
export WATER_MODEL="spce"
export USE_GPU="false"
export GPU_PARTITION="gpu"
export CPU_PARTITION="build"

# Show configuration
echo "=== Amber Workflow Configuration ==="
echo "Project:           $PROJECT_NAME"
echo "Input structure:   $INPUT_PDB"
echo "Work directory:    $WORK_DIR"
echo "MPI ranks:         $MPI_RANKS"
echo "Force field:       leaprc.protein.$FORCE_FIELD"
echo "Water model:       leaprc.water.$WATER_MODEL"
echo "Use GPU:           $USE_GPU"
echo ""
echo "✅ Configuration loaded. Use these variables in templates below."


### Directory and Input File Setup

All workflows require a leaprc script (system preparation) and three `.mdin` files (simulation parameters). Create them here:


In [ ]:
import os

# Energy minimization control file (em.mdin)
em_mdin = """Amber energy minimization
 &cntrl
  imin   = 1,          ! Perform minimization (not MD)
  maxcyc = 5000,       ! Max minimization steps
  ncyc   = 2500,       ! Steps of steepest descent before switching to CG
  ntb    = 1,          ! Periodic boundary (constant volume)
  ntp    = 0,          ! No pressure scaling during minimization
  cut    = 8.0,        ! Non-bonded cutoff (Angstroms)
  ntpr   = 100,        ! Print energy every 100 steps
  ntwr   = 5000,       ! Write restart every 5000 steps
 /
"""

# Heating control file (heat.mdin): NVT, 0 → 300 K, 50 ps
heat_mdin = """Amber NVT heating 0 to 300 K (50 ps)
 &cntrl
  imin   = 0,           ! Dynamics (not minimization)
  irest  = 0,  ntx = 1, ! New run: read coords only, assign random velocities
  nstlim = 25000,  dt = 0.002,  ! 50 ps total (25 000 steps x 2 fs)
  ntb    = 1,  ntp = 0,          ! Constant volume (NVT)
  ntt    = 3,  gamma_ln = 1.0,   ! Langevin thermostat (collision freq 1/ps)
  tempi  = 0.0,  temp0 = 300.0,  ! Heat from 0 K to 300 K
  ntpr   = 100,  ntwx = 100,  ntwr = 5000,
  cut    = 8.0,
  nmropt = 1,           ! Enable NMR restraints (used for temperature ramp)
 /
 &wt TYPE='TEMP0', istep1=0, istep2=25000, value1=0.0, value2=300.0, /
 &wt TYPE='END', /
"""

# Production MD control file (prod.mdin): NPT, 300 K / 1 bar, 100 ps
prod_mdin = """Amber NPT production MD (100 ps at 300 K / 1 bar)
 &cntrl
  imin   = 0,           ! Dynamics
  irest  = 1,  ntx = 5, ! Restart: read coords AND velocities from rst7
  nstlim = 50000,  dt = 0.002,   ! 100 ps total (50 000 steps x 2 fs)
  ntb    = 2,  ntp = 1,          ! Constant pressure (NPT)
  pres0  = 1.0,  taup = 2.0,     ! Target pressure 1 bar, coupling time 2 ps
  ntt    = 3,  gamma_ln = 1.0,   ! Langevin thermostat
  temp0  = 300.0,                ! Target temperature 300 K
  ntpr   = 100,  ntwx = 100,  ntwr = 5000,
  cut    = 8.0,
  iwrap  = 1,                    ! Wrap coords back into primary simulation box
 /
"""

# tleap script (leaprc.input): builds system.prmtop + system.inpcrd from the test PDB
# Force field: ff99SBildn (protein)   Water model: SPC/E
leaprc = """# Amber system preparation script
source leaprc.protein.ff99SBildn
source leaprc.water.spce

# Load structure
mol = loadpdb input/ala.pdb

# Inspect the system for missing atoms or parameter problems
check mol

# Add a 10 Angstrom SPC/E water shell
solvateBox mol SPCBOX 10.0

# Neutralize charge by adding counterions
addions mol Na+ 0

# Write Amber topology and initial coordinates
saveamberparm mol system.prmtop system.inpcrd

# Write PDB for visual inspection
savepdb mol system.pdb

quit
"""

os.makedirs('mdin_files', exist_ok=True)

with open('mdin_files/em.mdin', 'w') as f:    f.write(em_mdin)
with open('mdin_files/heat.mdin', 'w') as f:  f.write(heat_mdin)
with open('mdin_files/prod.mdin', 'w') as f:  f.write(prod_mdin)
with open('leaprc.input', 'w') as f:           f.write(leaprc)

print("✅ Created Amber input files:")
print("  • mdin_files/em.mdin    (energy minimization, 5,000 steps)")
print("  • mdin_files/heat.mdin  (NVT heating 0→300 K, 50 ps)")
print("  • mdin_files/prod.mdin  (NPT production MD, 100 ps)")
print("  • leaprc.input          (ff99SBildn + SPC/E water)")


**Expected output:**
```
✅ Created Amber input files:
  • mdin_files/em.mdin    (energy minimization, 5,000 steps)
  • mdin_files/heat.mdin  (NVT heating 0→300 K, 50 ps)
  • mdin_files/prod.mdin  (NPT production MD, 100 ps)
  • leaprc.input          (ff99SBildn + SPC/E water)
```


---

### Workflow 1: Single-Node CPU Simulation (Build Partition)

**Use when:** System < 10–50K atoms, no GPU access, testing or rapid iteration


In [ ]:
%%bash
# WORKFLOW 1: Single-Node CPU Simulation
# Runs tleap → sander (EM) → sander.MPI (heat) → sander.MPI (prod)
# Script: amber.sbatch  (loops over all PDB files in INPUT_DIR)

# Step 1: Verify both modules
module load amber ambertools 2>/dev/null
amber.python --version 2>/dev/null | head -2

# Step 2: Run tleap to build the test system
echo ""
echo "=== Running tleap to build system ==="
echo "(reads leaprc.input, writes system.prmtop + system.inpcrd)"
tleap -f leaprc.input 2>&1 | tail -12

# Step 3: Confirm output files
echo ""
echo "=== System files ready ==="
ls -lh system.prmtop system.inpcrd system.pdb 2>/dev/null || echo "NOTE: tleap not available — check: module load ambertools"

# Step 4: PRODUCTION SUBMISSION (uncomment the block below when ready)
# ⚠️ Warning: submits a multi-hour job to the build partition via amber.sbatch
#
# INPUT_DIR=./input OUTPUT_BASE=./output MDIN_DIR=./mdin_files LEAPRC=./leaprc.input \
# sbatch --partition=build --nodes=1 --ntasks=16 amber.sbatch


**Expected output (tleap run):**
```
Welcome to LEaP, version 24.x ...
Loading parameters: /amber/dat/leap/parm/parm10.dat
...
Building topology.
...
Exiting LEaP: Errors = 0; Warnings = 0; Notes = 1.

=== System files ready ===
-rw-r--r-- 1 user group 310K Apr  6 10:01 system.prmtop
-rw-r--r-- 1 user group  52K Apr  6 10:01 system.inpcrd
-rw-r--r-- 1 user group  21K Apr  6 10:01 system.pdb
```

### Workflow 1 Template: Single-Node CPU Simulation

**Use when:** System < 50K atoms, CPU partition available, quick turnaround needed

Submit using `amber.sbatch`:
```bash
INPUT_DIR=./input OUTPUT_BASE=./output MDIN_DIR=./mdin_files LEAPRC=./leaprc.input \
sbatch --partition=build --nodes=1 --ntasks=16 amber.sbatch
```

`amber.sbatch` will loop over every `.pdb` in `INPUT_DIR`, run `tleap → sander (EM) → sander.MPI (heat) → sander.MPI (prod)` for each, and save results under `OUTPUT_BASE/<basename>/`.

For custom SLURM parameters, override at submission time:
```bash
sbatch --partition=build --nodes=1 --ntasks=8 --time=04:00:00 --mem=16G amber.sbatch
```

**⚠️ Before submitting:**
1. Run `tleap -f leaprc.input` on the login node first to confirm the system builds cleanly
2. Check `system.prmtop` is non-zero and `tleap` output shows `Errors = 0`
3. Adjust `--ntasks` to match your system size (16 ranks is the practical ceiling for most proteins with `sander.MPI`)


---

### Workflow 2: GPU-Accelerated Simulation

**Use when:** System > 5K atoms, GPU nodes available, need 50–200x speedup over CPU


In [ ]:
%%bash
# WORKFLOW 2: GPU-Accelerated Simulation with pmemd.cuda
# Use when: Systems > 5K atoms, GPU nodes available, production runs
# Script: amber_gpu.sbatch  (takes pre-built prmtop + inpcrd as positional args)
# Modules: amber (pmemd.cuda) + ambertools (tleap, sander for minimization)

# Step 1: Verify both modules
module load amber ambertools 2>/dev/null
amber.python --version 2>/dev/null | head -2

echo ""
echo "=== GPU engine check ==="
if command -v pmemd.cuda &>/dev/null; then
    echo "✅ pmemd.cuda is available  (amber module)"
else
    echo "⚠️  pmemd.cuda not found — check: module load amber, and use a GPU node"
fi

echo ""
echo "=== GPU availability ==="
nvidia-smi -L 2>/dev/null || echo "No nvidia-smi (check on gpu partition nodes)"

echo ""
echo "=== Simulation stages for GPU workflow ==="
echo "  Stage 1: sander     (minimization — CPU, ambertools)"
echo "  Stage 2: pmemd.cuda (heating 0→300 K — GPU, amber)"
echo "  Stage 3: pmemd.cuda (production NPT, 100 ps — GPU, amber)"

# PRODUCTION SUBMISSION (uncomment when ready)
# ⚠️ Warning: submits a multi-hour GPU job via amber_gpu.sbatch
# Requires: system.prmtop + system.inpcrd already built by tleap
#
# sbatch --partition=gpu --gres=gpu:1 \
#   amber_gpu.sbatch system.prmtop system.inpcrd output_gpu


**Expected output:**
```
GPU engine check:
✅ pmemd.cuda is available  (amber module)

GPU availability:
GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-...)

Simulation stages for GPU workflow:
  Stage 1: sander     (minimization — CPU, ambertools)
  Stage 2: pmemd.cuda (heating 0→300 K — GPU, amber)
  Stage 3: pmemd.cuda (production NPT, 100 ps — GPU, amber)
```

### Workflow 2 Template: GPU-Accelerated Simulation

**Use when:** System > 5K atoms, GPU nodes available, need speed

Submit using `amber_gpu.sbatch`, passing your pre-built topology and coordinates as positional arguments:
```bash
sbatch --partition=gpu --gres=gpu:1 \
  amber_gpu.sbatch system.prmtop system.inpcrd output_gpu
```

`amber_gpu.sbatch` runs: `sander (EM, CPU) → pmemd.cuda (heat) → pmemd.cuda (prod, 100 ps)` and saves results under `output_gpu/`.

For custom SLURM parameters, override at submission time:
```bash
sbatch --partition=gpu --gres=gpu:1 --time=12:00:00 --mem=32G \
  amber_gpu.sbatch system.prmtop system.inpcrd output_gpu
```

**⚠️ Before submitting:**
1. Confirm `system.prmtop` and `system.inpcrd` exist (built by `tleap`)
2. Check GPU partition availability: `sinfo -t available -p gpu`
3. Energy minimization always runs on CPU (`sander`); heat and production on GPU (`pmemd.cuda`)
4. GPU utilization should be > 90% during production — verify with `nvidia-smi -l 1`

**GPU performance reference:**  
~100–200 ns/day for proteins of 10–50K atoms on a modern A100 GPU.


---

### Workflow 3: Multi-Node Production Run

**Use when:** Very large systems (> 100K atoms), long production runs, multi-node scaling needed


In [ ]:
%%bash
# WORKFLOW 3: Multi-Node Production Run with sander.MPI
# Use when: Very large systems (> 100K atoms), long production runs, multi-node needed
# Script: amber_mpi.sbatch  (takes pre-built prmtop + inpcrd as positional args)
# Modules: amber (version check) + ambertools (tleap, sander, sander.MPI)

# Step 1: Verify both modules
module load amber ambertools 2>/dev/null
amber.python --version 2>/dev/null | head -2

echo ""
echo "=== MPI & Interconnect Info ==="
which mpirun && mpirun --version 2>&1 | head -2

echo ""
echo "=== Parallelization for multi-node ==="
echo "Standard (1 node):  mpirun -np 16 sander.MPI -O ..."
echo "Multi-node (4×8):   mpirun -np 32 --mca btl_openib_allow_ib true sander.MPI -O ..."
echo ""
echo "SLURM environment:"
echo "  SLURM_NTASKS = total MPI ranks across all nodes"

echo ""
echo "=== Simulation stages for MPI workflow ==="
echo "  Stage 1: sander     (minimization — serial, ambertools)"
echo "  Stage 2: sander.MPI (heating 0→300 K — multi-node, ambertools)"
echo "  Stage 3: sander.MPI (production NPT, 100 ps — multi-node, ambertools)"

echo ""
echo "=== Cluster resources ==="
sinfo -N 2>/dev/null | head -10 || echo "(sinfo not available)"

# PRODUCTION SUBMISSION (uncomment when ready)
# ⚠️ Warning: submits a multi-hour multi-node job via amber_mpi.sbatch
# Requires: system.prmtop + system.inpcrd already built by tleap
#
# sbatch --nodes=4 --ntasks-per-node=8 \
#   amber_mpi.sbatch system.prmtop system.inpcrd output_mpi


**Expected output:**
```
MPI & Interconnect Info:
/usr/mpi/bin/mpirun
mpirun (Open MPI) 4.x.x ...

Parallelization for multi-node:
Standard (1 node):  mpirun -np 16 sander.MPI -O ...
Multi-node (4×8):   mpirun -np 32 --mca btl_openib_allow_ib true sander.MPI -O ...
```

### Workflow 3 Template: Multi-Node Production Run

**Use when:** Very large systems (> 100K atoms), production runs, multi-node needed

```bash
#!/bin/bash
#SBATCH --job-name=amber-multinode
#SBATCH --partition=build
#SBATCH --nodes=4
#SBATCH --ntasks-per-node=8
#SBATCH --cpus-per-task=1
#SBATCH --mem=64G
#SBATCH --time=24:00:00
#SBATCH --output=amber_multinode_%j.out
#SBATCH --error=amber_multinode_%j.err

# amber     → version check utility
# ambertools → tleap, sander, sander.MPI
module load amber ambertools

TOTAL_RANKS=$SLURM_NTASKS   # 4 nodes × 8 ranks = 32 MPI processes

# Stage 0: System preparation (tleap from ambertools — serial only)
tleap -f leaprc.input

# Stage 1: Energy minimization (serial sander from ambertools)
sander -O \
  -i mdin_files/em.mdin   -o em.mdout \
  -p system.prmtop -c system.inpcrd -r em.rst7

# Stage 2: Heating, multi-node (sander.MPI from ambertools)
mpirun -np $TOTAL_RANKS \
  --mca btl_openib_allow_ib true \
  sander.MPI -O \
  -i mdin_files/heat.mdin   -o heat.mdout \
  -p system.prmtop -c em.rst7 -r heat.rst7 -x heat.nc

# Stage 3: Production MD, multi-node (sander.MPI from ambertools)
mpirun -np $TOTAL_RANKS \
  --mca btl_openib_allow_ib true \
  sander.MPI -O \
  -i mdin_files/prod.mdin   -o prod.mdout \
  -p system.prmtop -c heat.rst7 -r prod.rst7 -x prod.nc \
  -inf prod.mdinfo

echo "✅ Multi-node Amber workflow complete!"
```

**Parallelization breakdown:**
- 4 nodes × 8 ranks/node = 32 MPI processes (all `sander.MPI` from ambertools)
- `tleap` always runs serially (single rank, ambertools)
- Energy minimization runs serially (`sander`, ambertools)

**⚠️ Before running:**
1. Verify multi-node access: `sinfo -t available | grep build`
2. Adapt `--ntasks-per-node`: Match cores per physical node
3. Update MPI transport: `--mca btl_openib_allow_ib true` for InfiniBand
4. For very large systems (> 500K atoms): Consider GPU+MPI with `pmemd.MPI.cuda` (requires `amber` module + GPU nodes)
5. Submit: Uncomment the heredoc block in the cell above, then run it

**Checkpoint Strategy (Important for long runs):**
- `ntwx` and `ntwr` in `.mdin` control trajectory and restart write frequency
- If job hits time limit: Resubmit with `irest=1, ntx=5` and the last `.rst7` file
- Use the last `prod.rst7` as the new `-c` input coordinate file


---

## PART IV — Troubleshooting (Skim When Broken)

**Purpose:** "Something failed. Give me answers fast."


---

### Monitoring Your Job


In [ ]:
%%bash
echo "=== 1. Check Current Job Status ==="
squeue -u $USER --format="%.7i %.20j %.2t %.10M %.6D %R"

echo ""
echo "=== 2. Get Partition Info ==="
sinfo | grep -E "(gpu|build|cpu)" | head -5

echo ""
echo "=== 3. View Completed Jobs ==="
sacct -u $USER --format=JobID,JobName,State,Elapsed | tail -5


**Expected output:**
```
=== 1. Check Current Job Status ===
  JOBID                 NAME  ST       TIME  NODES
  12345        amber-cpu-run   R       5:32      1

=== 2. Get Partition Info ===
  build  up  8:00:00  128  idle  node[01-16]
  gpu    up 24:00:00    8  idle  gpunode[01-08]
```

---

### Comprehensive diagnostics — checks both modules

In [ ]:
%%bash

module load amber ambertools 2>/dev/null

echo "=== Amber Environment Diagnostics ==="
echo ""

echo "1. ambertools binaries (tleap, sander, cpptraj)"
command -v tleap      &>/dev/null && echo "  ✅ tleap:       $(which tleap)"      || echo "  ⚠️  tleap not in PATH    — check: module load ambertools"
command -v sander     &>/dev/null && echo "  ✅ sander:      $(which sander)"     || echo "  ⚠️  sander not in PATH   — check: module load ambertools"
command -v sander.MPI &>/dev/null && echo "  ✅ sander.MPI:  $(which sander.MPI)" || echo "  ⚠️  sander.MPI not in PATH — check: module load ambertools"
command -v cpptraj    &>/dev/null && echo "  ✅ cpptraj:     $(which cpptraj)"    || echo "  ⚠️  cpptraj not in PATH  — check: module load ambertools"
echo ""

echo "2. amber binaries (pmemd.cuda)"
command -v pmemd.cuda &>/dev/null && echo "  ✅ pmemd.cuda:  $(which pmemd.cuda)" || echo "  ⚠️  pmemd.cuda not in PATH (needs GPU node + module load amber)"
echo ""

echo "3. MPI Library"
command -v mpirun &>/dev/null && echo "✅ MPI: $(mpirun --version 2>&1 | head -1)" || echo "⚠️  MPI not in PATH"
echo ""

echo "4. GPU Detection"
if command -v nvidia-smi &>/dev/null; then
    echo "✅ NVIDIA GPU found:"
    nvidia-smi --query-gpu=index,name,memory.total --format=csv,noheader 2>/dev/null
else
    echo "⚠️  No GPU detected (GPU available only on gpu partition)"
fi
echo ""

echo "5. Available Partitions"
sinfo -h --format="%.10P %.5D %.14T %.11l" 2>/dev/null | head -10 || echo "(Partition info not available; use 'sinfo')"


In [ ]:
%%bash

echo "=== Recent Job Status (Last 10 Jobs) ==="
sacct -u $USER --format=JobID,JobName,State,Elapsed,ExitCode --starttime=2026-01-01 2>/dev/null | tail -11 ||   echo "(Job history not available)"
echo ""

echo "=== Most Recent SLURM Log (If Job Failed) ==="
if ls slurm-*.out 2>/dev/null | head -1 | grep -q .; then
    latest_log=$(ls -t slurm-*.out | head -1)
    echo "Checking: $latest_log"
    echo ""
    tail -30 "$latest_log" 2>/dev/null | head -20
else
    echo "No SLURM logs found in current directory"
fi
echo ""

echo "=== Most Recent Amber Output Log ==="
if ls *.mdout 2>/dev/null | head -1 | grep -q .; then
    latest_mdout=$(ls -t *.mdout | head -1)
    echo "Checking: $latest_mdout"
    tail -25 "$latest_mdout" 2>/dev/null
else
    echo "No .mdout files found in current directory"
fi


---

### Common Errors and Fixes

| Error | Likely Cause | Fix |
|-------|-------------|-----|
| `sander: command not found` | Module not loaded | `module load amber` |
| `pmemd.cuda: command not found` | On CPU node or module issue | Submit to `--partition=gpu`; `module load amber` |
| `forrtl: error (78): process killed` | OOM — job ran out of memory | Increase `#SBATCH --mem` or reduce system size |
| `FATAL: Mismatched domain in prmtop` | Wrong topology for system | Re-run `tleap -f leaprc.input` |
| `CUDA error: out of memory` | System too large for GPU RAM | Use fewer atoms or switch to `sander.MPI` |
| `Ewald: sum did not converge` | Unstable starting geometry | Re-run minimization with more `maxcyc` steps |
| `NaN in energy` | Simulation blowing up | Check initial structure; run minimization first |
| `Job stuck in PD` | No resources available | Reduce `--nodes` / `--ntasks` or wait |
| `UCX/MPI abort` | Network issue on multi-node | Add `--mca btl_openib_allow_ib true` to mpirun |


---

### Understanding Output Files

After a successful Amber run, look in your working directory:

| File | Description | Check For |
|------|-------------|----------|
| `.prmtop` | Amber topology (from tleap) | Exists and non-zero size |
| `.inpcrd` / `.rst7` | Coordinates / restart file | Final coords for next stage |
| `.mdout` | Simulation log | Energies, temperature, pressure vs time |
| `.nc` / `.mdcrd` | Trajectory (NetCDF) | Atomic positions over time (main output) |
| `.mdinfo` | Running summary | Updated each step; shows ns/day performance |

**Quick sanity check — run after any stage completes:**
```bash
# Check that the restart file was written (non-zero size)
ls -lh *.rst7

# Check the last few lines of the energy log
tail -20 prod.mdout | grep -E "NSTEP|Etot|Temp|Press|Density"

# Verify trajectory was written
ls -lh prod.nc
```


---

### FAQ

**Q: What is the difference between `sander` and `pmemd.cuda`?**

A: `sander` is the original CPU-based engine (serial or MPI), included free in AmberTools. `pmemd.cuda` is the GPU-accelerated engine from the licensed Amber package. `pmemd.cuda` is 50–200x faster for proteins > 5K atoms, but requires both an Amber license and a GPU node.

**Q: How do I restart a simulation from a checkpoint?**

A: Use the last `.rst7` file as the new coordinate input. Change your `mdin` to `irest=1, ntx=5` (restart with velocities):
```bash
sander -O -i mdin_files/prod.mdin -o prod_cont.mdout \
  -p system.prmtop -c prod.rst7 -r prod_cont.rst7 -x prod_cont.nc
```

**Q: How do I check my simulation's performance (ns/day)?**

A: Inspect the `Performance` section at the end of `.mdout`:
```bash
grep "ns/day" prod.mdout
```
Or watch it live via: `tail -f prod.mdinfo | grep Simultime`

**Q: How much disk space do I need?**

A: Rule of thumb: 10× the topology size per stage.
- `system.prmtop` = 2 MB → expect ~20 MB of output per simulation stage
- 1 μs trajectory (50K atoms, ntwx=500): ~5–50 GB in NetCDF format

**Q: Why does my GPU job run slower than my CPU job?**

A: Common causes:
1. **System too small** (< 5K atoms): GPU overhead dominates
2. **pmemd.cuda not in PATH**: Check `which pmemd.cuda` on the compute node
3. **GPU not being used**: Run `nvidia-smi` during job; look for `pmemd.cuda` process

**Q: Can I run multiple proteins at once?**

A: Yes — submit one SLURM job per system. Each job is independent.

**Q: How do I extend a simulation that ran out of wall time?**

A: Resubmit with the last `.rst7` as input coordinates, with `irest=1, ntx=5` in the `.mdin`. No data is lost; the trajectory simply picks up from where `ntwx` last wrote frames.

**Q: Where do I get help?**

A: 
- **Amber Manual**: https://ambermd.org/doc12/Amber25.pdf
- **Official Tutorials**: https://ambermd.org/tutorials/
- **Mailing List**: https://lists.ambermd.org/mailman/listinfo/amber
- **Local support**: Contact your HPC system administrator


---

## APPENDIX A — Best Practices for Amber Simulations

**Purpose:** "Prevent common failures and get science-grade results."

### System Preparation (Pre-Simulation)

1. **Verify structure quality before tleap**
   - Check for missing atoms, alternate conformations, and chain breaks
   - Use WHATCHECK, MolProbity, or PDB-Care for quality assessment
   - Fix issues in your PDB editor before loading into tleap

2. **Choose force field for your scientific question**
   - `ff14SB` (protein): Most common; good for folded proteins
   - `ff19SB` (protein): Improved backbone torsions; newer alternative
   - `GAFF2` (small molecules): Use with antechamber for ligands
   - `ff14SB + GAFF2` (protein-ligand): Standard for drug binding studies

3. **Solvate with adequate water shell**
   - Minimum 10 Å on all sides: `solvateBox mol TIP3PBOX 10.0`
   - Smaller box = artificial artifacts from periodic images
   - Larger box = more water atoms = slower simulation

4. **Always add counterions to neutralize net charge**
   - `addions mol Na+ 0` or `addions mol Cl- 0`
   - Charged system = non-physical electrostatics in PBC
   - Record net charge and final ion count in your lab notebook

5. **Always run energy minimization before any MD**
   - Minimum 2000–5000 steps (`maxcyc = 5000`)
   - Use `sander` (serial) for minimization — more robust than sander.MPI
   - Prevents "simulation blew up" errors in the first picoseconds

### Production Run Execution

6. **Equilibrate in NVT before NPT**
   - Phase 1: NVT heating 0→300 K (heat.mdin), 50–100 ps minimum
   - Phase 2: NPT equilibration at 300 K / 1 bar, 100–500 ps minimum
   - Phase 3: Production NPT (prod.mdin, your actual data)
   - Skipping equilibration biases thermodynamic and structural results

7. **Monitor energy conservation during the first nanosecond**
   - Check temperature, pressure, and density in `.mdout`
   - Large fluctuations in the first ~100 ps are normal; should stabilize
   - If density drifts continuously: check compressibility settings in `.mdin`

8. **Save restart files frequently**
   - `ntwr = 5000` (saves every 10 ps at 2 fs timestep)
   - Enables seamless restarts if job hits wall time or node fails
   - Restart files are small (~few MB) — keep them all

9. **Use 2 fs timestep for standard protonated systems**
   - `dt = 0.002` with hydrogen mass repartitioning (HMR) allows 4 fs
   - HMR: add `HMassRepartition` to tleap to redistribute H mass
   - 4 fs timestep = 2× longer simulation for same wall time

10. **Use pmemd.cuda whenever a GPU is available**
    - 50–200× speedup for systems > 5K atoms
    - Always run minimization on CPU (`sander`) — more stable
    - Heat and produce on GPU (`pmemd.cuda`)


---

## APPENDIX B — Recommended Project Layout

Organize your Amber projects with this directory structure for easy restarts and reproducibility:

```
amber_project/
├── structures/                     # Input PDB files and tleap scripts
│   ├── protein.pdb                # Original deposited structure
│   ├── ligand.mol2                # Small molecule input (for antechamber)
│   ├── leaprc.input               # Master tleap script
│   ├── ligand.gaff.frcmod         # GAFF parameters from antechamber
│   └── README_STRUCTURES.txt      # Source, chain IDs, protonation notes
│
├── parameters/                     # mdin files and topology
│   ├── em.mdin                    # Energy minimization
│   ├── heat.mdin                  # NVT heating 0→300 K
│   ├── eqnpt.mdin                 # NPT equilibration
│   ├── prod.mdin                  # Production NPT run
│   └── README_MDP.txt             # mdin settings rationale
│
├── runs/                          # Simulation outputs (organized by date/attempt)
│   ├── 2026_04_test/
│   │   ├── em/                    # Energy minimization output
│   │   │   ├── em.mdout
│   │   │   └── em.rst7            ← input for heating
│   │   ├── heat/
│   │   │   ├── heat.mdout
│   │   │   ├── heat.rst7          ← input for production
│   │   │   └── heat.nc
│   │   └── prod/                  # Production run (main output)
│   │       ├── prod.mdout
│   │       ├── prod.nc            ← trajectory (main output)
│   │       ├── prod.rst7          ← restart point
│   │       └── prod.mdinfo
│   └── README_RUNS.txt            # Log of what was run when
│
├── analysis/                      # Post-simulation analysis
│   ├── cpptraj/                   # RMSD, clustering, RMSF scripts + outputs
│   ├── mmpbsa/                    # Binding free energy
│   └── scripts/
│       ├── analyze_rmsd.sh
│       └── run_mmpbsa.sh
│
├── scripts/                       # SLURM submission scripts
│   ├── submit_cpu.sh
│   ├── submit_gpu.sh
│   └── submit_multinode.sh
│
└── data/                          # Experimental reference data for validation
    ├── exp_structure.pdb
    └── exp_README.txt
```

**Why this layout?**
- **Clarity**: Each directory has a single, well-defined purpose
- **Restartability**: All inputs are co-located with the stage that produced them
- **Reproducibility**: Scripts and parameters are version-controlled separately
- **Scalability**: Multiple runs (by date or parameter set) coexist without conflict


---

## APPENDIX C — Amber File Types Reference

### Input Files

| Extension | Full Name | Purpose | Required? |
|-----------|-----------|---------|---|
| `.pdb` | Protein Data Bank | Starting atomic coordinates | ✅ Yes |
| `.prmtop` | Amber topology | Atom types, bonds, angles, force-field parameters (from tleap) | ✅ Yes |
| `.inpcrd` | Initial coordinates | Starting positions (from tleap) | ✅ Yes |
| `.rst7` | Restart file | Coordinates + velocities for restart | For heating/production |
| `.mdin` | MD input | Simulation control parameters (integrator, temperature, cutoffs, etc.) | ✅ Yes |
| `.mol2` | Tripos MOL2 | Small molecule structure for antechamber | For ligands |
| `.frcmod` | Force field modification | Custom parameters for non-standard residues | For ligands |

### Output Files (From Simulations)

| Extension | Full Name | Purpose | Important? |
|-----------|-----------|---------|---|
| `.mdout` | MD output log | Step-by-step energies, temperature, pressure | ✅ Critical |
| `.rst7` | Restart file | Final coordinates + velocities; input for next stage | ✅ Save all |
| `.nc` | NetCDF trajectory | Compressed atomic positions over time | ✅ Main output |
| `.mdcrd` | MD coordinates | Uncompressed trajectory (large; prefer `.nc`) | ⚠️ Avoid |
| `.mdinfo` | MD info | Running summary updated in real time | Optional |

### File Size Rule of Thumb

```
1 microsecond simulation, 50,000 atoms:
├── .nc (NetCDF, compressed)  ≈ 5–50 GB  ← USE THIS (ntwx=500)
├── .mdcrd (uncompressed)     ≈ 50+ GB   ← AVOID
├── .mdout (energy log)       ≈ 50 MB
├── .rst7 (restart)           ≈ 5 MB
└── Total expected            ≈ 50–100 GB (with .nc)
```

**Key Insight:** Always write `.nc` (NetCDF, controlled by `ntwx` in `.mdin`). The `ntwr` parameter controls restart file frequency—set to at minimum every 50,000 steps.


---

## APPENDIX D — Storage and Resource Requirements

### Disk Space Planning

**Per-simulation storage needs:**

| System Size | Duration | Format | Size | Notes |
|---|---|---|---|---|
| 5K atoms | 100 ns | `.nc` | 500 MB | Small test system |
| 5K atoms | 1 μs | `.nc` | 5 GB | Typical small system |
| 50K atoms | 100 ns | `.nc` | 5 GB | Medium protein in water |
| 50K atoms | 1 μs | `.nc` | 50 GB | Production run |
| 100K atoms | 100 ns | `.nc` | 10 GB | Large protein complex |
| 100K atoms | 1 μs | `.nc` | 100 GB | Multi-node production |

### Memory and CPU Resources

| System Size | CPU Ranks | GPU RAM | Time Estimate (GPU) | Typical SLURM Request |
|---|---|---|---|---|
| 5K atoms | 4–8 | Not recommended | ~200 ns/day | `--ntasks=8` |
| 5K atoms (sander.MPI) | 4–16 | N/A | ~5 ns/day | `--ntasks=8 --cpus-per-task=1` |
| 50K atoms | 1 | 8–16 GB | ~100 ns/day | `--ntasks=1 --gres=gpu:1` |
| 50K atoms (CPU only) | 16–32 | N/A | ~2 ns/day | `--ntasks=16 --cpus-per-task=1` |
| 100K atoms | 1 | 24–40 GB | ~30 ns/day | `--ntasks=1 --gres=gpu:1` |
| > 200K atoms | 32–64 | 40+ GB | ~10 ns/day | Multi-node or multi-GPU |

### Common Wall-Time Estimates

| Use Case | Partition | Time Request | Notes |
|---|---|---|---|
| Testing (1 ns, ≤5K atoms) | build | `1:00:00` | Quick debugging |
| Small CPU run (50 ns, 5K atoms) | build | `12:00:00` | ~5 ns/day |
| GPU production (500 ns, 50K atoms) | gpu | `48:00:00–72:00:00` | ~100 ns/day |
| Large GPU run (100 ns, 100K atoms) | gpu | `72:00:00` | ~30 ns/day |
| Analysis only | build | `2:00:00` | cpptraj is CPU-only |

### Never Over-Request Resources

- ❌ Requesting 64 CPU cores for a 5K atom system (sander.MPI tops out at ~16 ranks)
- ❌ Requesting a GPU for minimization (use `sander`, not `pmemd.cuda`)
- ❌ Requesting 500 GB RAM when 32 GB suffices
- ✅ Use checkpointed restarts for runs > 48 hours
- ✅ Profile on a 1 ns test run before committing to microsecond production


---

## APPENDIX E — Amber Documentation, Tutorials, and Resources

### Official Amber Documentation

**Main Resources:**
- 📖 **Amber Reference Manual (PDF)**: https://ambermd.org/doc12/Amber25.pdf (complete reference)
- 🎓 **Official Tutorials**: https://ambermd.org/tutorials/ (step-by-step guides)
- 💻 **Amber Website**: https://ambermd.org/ (news, downloads, force fields)
- 📧 **Mailing List**: https://lists.ambermd.org/mailman/listinfo/amber (active community)

### Local Documentation (This System)


In [ ]:
%%bash
# Find local Amber documentation
# amber.python is provided by ambertools
module load amber ambertools 2>/dev/null

echo "=== Local Amber Documentation ==="
echo ""

AMBER_HOME=$(amber.python -c "import os; print(os.environ.get('AMBERHOME','not-set'))" 2>/dev/null || echo "not-set")

if [ "$AMBER_HOME" != "not-set" ] && [ -d "$AMBER_HOME" ]; then
    echo "✅ Amber installation found:"
    echo "   AMBERHOME: $AMBER_HOME"
    echo ""

    if [ -d "$AMBER_HOME/doc" ]; then
        echo "📚 Documentation directory:"
        ls -lh "$AMBER_HOME/doc/" 2>/dev/null | head -10
    fi

    if [ -d "$AMBER_HOME/test" ]; then
        echo ""
        echo "🧪 Test suite directory (runnable examples):"
        ls "$AMBER_HOME/test/" 2>/dev/null | head -15
    fi
else
    echo "⚠️  AMBERHOME not set"
    echo "    Try: module load amber ambertools && echo \$AMBERHOME"
fi

echo ""
echo "=== Quick Command-Line Help ==="
sander -h 2>&1 | head -15


### High-Quality Tutorials and Learning Resources

**Beginner-Friendly:**
- 🎬 **Amber Getting Started**: https://ambermd.org/tutorials/basic/
- 📄 **Tutorial 1: Solvated Protein** (alanine dipeptide): https://ambermd.org/tutorials/basic/tutorial0/index.php
- 📄 **Tutorial 4: Antechamber/GAFF** (small molecule parameters): https://ambermd.org/tutorials/basic/tutorial4b/index.php

**Intermediate to Advanced:**
- ⚙️ **Tutorial 3: MM-PBSA Binding Free Energy**: https://ambermd.org/tutorials/advanced/tutorial3/index.php
- 🧬 **Tutorial 21: Binding Enthalpy Calculations**: https://ambermd.org/tutorials/advanced/tutorial21/index.php
- 📊 **CPPTRAJ Analysis Guide**: https://amberhub.chpc.utah.edu/cpptraj-tutorials/

### Scientific References

**Primary Amber Citation:**
D.A. Case et al. *Amber 2024*. University of California, San Francisco.

**For Specific Modules:**
- `sander`: Salomon-Ferrer R, et al. *WIREs Comput. Mol. Sci.* 2013, 3, 198–210
- `pmemd.cuda`: Götz A.W., et al. *J. Chem. Theory Comput.* 2012, 8, 1542–1555
- `cpptraj`: Roe D.R. & Cheatham T.E. *J. Chem. Theory Comput.* 2013, 9, 3084–3095
- MM-PBSA: Homeyer N. & Gohlke H. *Mol. Inform.* 2012, 31, 114–122

### Integration with This Notebook

This Jupyter notebook **complements** the official Amber manual by providing:

✅ **Getting Started** (Part I): Quick 5-minute setup specific to this HPC system  
✅ **HPC Cluster Guidance** (Part II): Understanding sander vs pmemd and resource allocation  
✅ **Practical Workflows** (Part III): Copy-ready SLURM templates for CPU, GPU, and multi-node  
✅ **Troubleshooting** (Part IV): Common HPC-specific errors and diagnostics  
✅ **Best Practices** (Appendix A): Lessons from production simulation experience  

📚 **For force field theory and method details** → https://ambermd.org/doc12/Amber25.pdf  
🎓 **For detailed step-by-step tutorials** → https://ambermd.org/tutorials/  
🐛 **For bug reports or questions** → amber mailing list at lists.ambermd.org  
